# 4. Detect strong but cancelling topographic features

Candidates combine high local-gradient exposure with low vector coherence. This detects heterogeneous bathymetry without pretending that opposing gradients define a single net direction.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
for path in (ANALYSIS_ROOT, HERE):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
import footprint_tools as ft
sns.set_theme(style="whitegrid", context="notebook")
palette = {"AE":"#c44e52", "CE":"#4c72b0"}

data = ft.load_cache()
filled = data[(data.footprint_kind == "Filled") & (data.pv_averaging == "nonlinear")].copy()
score = ft.footprint_scorecard(data)

paths=tilt.Paths()
grid=tilt.load_grid(paths.grid,paths.z_r)


In [ ]:
full = filled[filled.footprint.eq("filled_1")].copy()
threshold = full.PV_grad_topo_p90_local_mag.quantile(.9)
full["feature_candidate"] = (full.PV_grad_topo_p90_local_mag >= threshold) & (full.PV_grad_topo_coherence < .35)
fig, axes = plt.subplots(1,2,figsize=(12,4.8),constrained_layout=True)
for ax,cyc in zip(axes,["AE","CE"]):
    part=full[full.Cyc.eq(cyc)]
    hb=ax.hexbin(part.PV_grad_topo_mean_local_mag,part.PV_grad_topo_coherence,gridsize=40,mincnt=1,bins="log",cmap="viridis")
    ax.axhline(.35,color="tab:red",ls="--"); ax.axvline(threshold,color="tab:red",ls="--")
    ax.set(xscale="log",xlabel="Mean local topographic magnitude",ylabel="Coherence",title=cyc)
fig.suptitle("Strong exposure with weak net direction identifies cancellation"); plt.show()

In [ ]:
def map_candidate(row):
    pad=max(50,1.4*row.Rc)
    inside=((grid.X_grid>=row.xc-pad)&(grid.X_grid<=row.xc+pad)&(grid.Y_grid>=row.yc-pad)&(grid.Y_grid<=row.yc+pad))
    bathy=np.where((grid.mask_rho==1)&inside,grid.h/1000,np.nan)
    fig,ax=plt.subplots(figsize=(6,5.5),constrained_layout=True)
    cf=ax.contourf(grid.X_grid,grid.Y_grid,bathy,levels=25,cmap="terrain_r")
    tilt.plot_ellipse(ax,row,grid,frac=1,color="white",lw=2)
    ax.scatter(row.xc,row.yc,c="magenta",s=35,zorder=10)
    ax.set(xlim=(row.xc-pad,row.xc+pad),ylim=(row.yc-pad,row.yc+pad),aspect="equal",
           title=(f"{row.Cyc}{int(row.Eddy)}, day {int(row.Day)}; "
                  f"coherence={row.PV_grad_topo_coherence:.2f}"))
    fig.colorbar(cf,ax=ax,label="Bathymetric depth (km)"); plt.show()

ranked=full[full.feature_candidate].sort_values("PV_grad_topo_p90_local_mag",ascending=False)
for _,row in ranked.groupby("Cyc",group_keys=False).head(3).iterrows():
    map_candidate(row)

Maps are a mandatory audit. Reject land-boundary artefacts, grid-edge cases and isolated single-cell gradients before interpreting a candidate as seamount interaction.